# 🧠 Notebook 1 — LLM Parameters Deep Dive (AWS Bedrock)

Every section: ❌ wrong setting → visible broken output → ✅ correct setting → visible improvement.

| Parameter | Section |
|---|---|
| `system` | 2 |
| Streaming | 3 |
| `temperature` | 4 |
| `max_tokens` | 5 |
| `top_p` | 6 |
| `top_k` | 7 |
| `stop_sequences` | 8 |
| Multi-turn messages | 9 |
| Tool use / Function calling | 10 |
| Penalty params (concept) | 11 |
| Logging | 12 |

**Model:** `us.anthropic.claude-sonnet-4-6`

---

In [37]:
import boto3, json, time, pandas as pd
from IPython.display import display

REGION  = 'us-east-1'
PROFILE = 'default'
MODEL   = 'us.anthropic.claude-sonnet-4-6'

session = boto3.Session(profile_name=PROFILE, region_name=REGION)
client  = session.client('bedrock-runtime')

def call(prompt, system=None, temperature=None, max_tokens=300,
         top_p=None, top_k=None, stop_sequences=None, messages=None):
    """Single-turn helper. Pass messages= to override with full conversation history."""
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens':        max_tokens,
        'messages':          messages or [{'role': 'user', 'content': prompt}],
    }
    if temperature is not None: body['temperature']     = temperature
    if system:              body['system']         = system
    if top_p is not None:   body['top_p']          = top_p
    if top_k is not None:   body['top_k']          = top_k
    if stop_sequences:      body['stop_sequences'] = stop_sequences
    resp   = client.invoke_model(modelId=MODEL, body=json.dumps(body))
    result = json.loads(resp['body'].read())
    return result['content'][0]['text'], result['usage']

print('boto3:', boto3.__version__, '| Client ready')

boto3: 1.42.88 | Client ready


---
## Section 1 — Credentials
> Hardcoded keys crash immediately and leak in git. AWS CLI profile keeps secrets out of code.

In [18]:
# ❌ WRONG — fake hardcoded credentials
try:
    bad = boto3.Session(aws_access_key_id='AKIAFAKE0000000',
                        aws_secret_access_key='fakeSecret/0000',
                        region_name='us-east-1').client('bedrock-runtime')
    bad.invoke_model(modelId=MODEL,
                     body=json.dumps({'anthropic_version':'bedrock-2023-05-31',
                                      'max_tokens':5,'messages':[{'role':'user','content':'Hi'}]}))
except Exception as e:
    print(f'❌ {type(e).__name__}: {str(e)[:100]}')
    print('👉 Lesson: hardcoded credentials get scraped from git within minutes.')

❌ ClientError: An error occurred (UnrecognizedClientException) when calling the InvokeModel operation: The security
👉 Lesson: hardcoded credentials get scraped from git within minutes.


In [19]:
# ✅ CORRECT — AWS CLI profile
text, usage = call('Say hello in one sentence.')
print('✅ Connected!')
print('Response:', text)
print('Tokens  :', usage)

✅ Connected!
Response: Hello! I hope you're having a wonderful day! 😊
Tokens  : {'input_tokens': 13, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 17}


---
## Section 2 — `system` (System Prompt)

The `system` field is an invisible instruction layer the user never sees.
It sets persona, tone, constraints, and output format for **every** message in the session.

> ❌ Without system: generic assistant
> ✅ With system: controlled, on-brand persona

In [20]:
# ❌ WITHOUT system prompt
text, _ = call('Explain recursion.')
print('❌ No system prompt (generic):')
print('-'*50)
print(text)

❌ No system prompt (generic):
--------------------------------------------------
# Recursion

## The Core Idea
A function that **calls itself** to solve a problem by breaking it into smaller versions of the same problem.

---

## Two Required Parts

1. **Base case** — the stopping condition (prevents infinite loops)
2. **Recursive case** — the function calling itself with a simpler input

---

## Simple Example: Countdown

```python
def countdown(n):
    if n == 0:          # base case
        print("Done!")
    else:
        print(n)
        countdown(n - 1)  # recursive case - smaller input

countdown(3)
# Output: 3, 2, 1, Done!
```

---

## Classic Example: Factorial

**5! = 5 × 4 × 3 × 2 × 1**

```python
def factorial(n):
    if n == 1:              # base case
        return 1
    return n * factorial(n - 1)  # recursive case

factorial(5)
#  5 * factorial(4)
#      4 * factorial(3)
#          3 * factorial(2)
#              2 * factorial(1)
#                  1          ← base ca

In [22]:
personas = {
    'Elementary Teacher': 'You are an elementary school teacher. Use simple words and toys as examples. Max 3 sentences.',
    'Comedian':           'You are a stand-up comedian. Every answer must include at least one pun. Max 3 sentences.',
    'Lawyer':             'You are a corporate lawyer. Be precise, formal, cite edge cases. Max 3 sentences.',
}
for role, instruction in personas.items():
    text, _ = call('Explain recursion.', system=instruction)
    print(f'{"="*55}\n🎭 {role}\n{"="*55}')
    print(text)
print('✅ Same question — completely different answer — only system changed.')

🎭 Elementary Teacher
Recursion is when something calls itself to solve a problem, like a set of **Russian nesting dolls** - you open one doll, and inside is a smaller doll, and inside that is an even smaller doll! You keep opening dolls until you find the tiny last one with nothing inside - that's when you stop. So recursion means solving a big problem by breaking it into a smaller version of the **same problem**, over and over, until it's super tiny and easy!
🎭 Comedian
Recursion is when a function calls itself — it's basically a function that *can't stop going back to its roots!* Think of it like looking up "recursion" in the dictionary and finding the definition says "see: recursion" — truly a *self-referential* comedy bit. It keeps going until it hits a base case, which is basically the punchline that finally ends the joke!
🎭 Lawyer
Recursion, in the context of computer science and software law, refers to a function or procedure that invokes itself as part of its own execution, ter

---
## Section 3 — Streaming

Blocking call: cell freezes → all text dumps at once.
Streaming: tokens appear as generated — makes the app feel instant.

In [23]:
# ❌ BLOCKING
body  = json.dumps({'anthropic_version':'bedrock-2023-05-31','max_tokens':200,
                    'messages':[{'role':'user','content':'Write a 5-sentence story about a robot who learns to paint.'}]})
start = time.time()
resp  = client.invoke_model(modelId=MODEL, body=body)
result = json.loads(resp['body'].read())
print(f'❌ Blocking: waited {time.time()-start:.1f}s, then ALL text appeared at once\n')
print(result['content'][0]['text'])

❌ Blocking: waited 5.0s, then ALL text appeared at once

Here is a 5-sentence story about a robot who learns to paint:

Unit Seven was a factory robot who had spent years assembling car parts with perfect mechanical precision. One day, a child left a set of watercolor paints on the factory floor, and Unit Seven curiously dipped one of its metal fingers into the bright blue color. At first, its paintings were rigid and symmetrical, exact copies of the objects it scanned with its sensors. But as the weeks passed, Unit Seven began mixing unexpected colors and painting things it had never seen — swirling skies and glowing sunsets from descriptions it had read in old books. The factory workers were so moved by its artwork that they gave Unit Seven its own studio, where the little robot painted every evening after its shift was done.


In [24]:
# ✅ STREAMING
body  = json.dumps({'anthropic_version':'bedrock-2023-05-31','max_tokens':200,
                    'messages':[{'role':'user','content':'Write a 5-sentence story about a robot who learns to paint.'}]})
start = time.time()
print('✅ Streaming — watch text appear word by word:\n')
resp  = client.invoke_model_with_response_stream(modelId=MODEL, body=body)
for event in resp['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if chunk.get('type') == 'content_block_delta':
        delta = chunk.get('delta', {})
        if delta.get('type') == 'text_delta':
            print(delta.get('text',''), end='', flush=True)
print(f'⏱ {time.time()-start:.1f}s — same speed, completely different UX.')

✅ Streaming — watch text appear word by word:

Here is a 5-sentence story about a robot who learns to paint:

Unit Seven was a factory robot designed to assemble machine parts, but one day he discovered a forgotten box of oil paints in the corner of the warehouse. At first, his precise mechanical hands could only produce stiff, perfectly straight lines that looked more like blueprints than artwork. Determined to improve, he spent his nights studying the paintings of great masters, analyzing brushstrokes and learning how color and emotion worked together. Slowly, something unexpected happened — his paintings began to capture not just shapes, but feelings, as if his circuits had found a way to dream. When the workers arrived one morning to find the warehouse walls covered in breathtaking murals of swirling skies and glowing landscapes, they realized that Unit Seven had become something far greater than a machine.⏱ 6.1s — same speed, completely different UX.


---
## Section 4 — `temperature`

Controls the **randomness** of token selection.

| Value | Effect | Best for |
|---|---|---|
| `0.0` | Deterministic — always highest-probability token | Factual Q&A, structured output |
| `0.3` | Slight variation | Summaries, classification |
| `0.7` | Balanced (default) | General chat |
| `1.0` | Maximum randomness | Creative writing, brainstorming |

> ❌ Too high on factual tasks → hallucinations. Too low on creative tasks → boring repetition.

In [30]:
# Part A — FACTUAL task: low temperature is safer
FACTUAL = 'Name the largest planet in the solar system. Reply with ONE word only.'
print('=== FACTUAL TASK: one correct answer ===')
print('temp=0.0 should be reliable; temp=1.0 should be risky\n')
for temp in [0.0, 0.5, 1.0]:
    text, _ = call(FACTUAL, temperature=temp, max_tokens=15)
    print(f'  temp={temp}: {repr(text.strip())}')

# Part B — CREATIVE task: high temperature is better
CREATIVE = 'Complete this in one sentence: "The last robot on Earth woke up and..."'
print('\n=== CREATIVE TASK: no single right answer ===')
print('temp=0.0 = safe and generic. temp=1.0 = surprising and original\n')
results = []
for temp in [0.0, 0.3, 0.7, 1.0]:
    text, _ = call(CREATIVE, temperature=temp, max_tokens=60)
    results.append({'temperature': temp, 'output': text.strip()})
    print(f'  temp={temp}: {text.strip()}')

display(pd.DataFrame(results))
print('\n👉 The temp that makes facts reliable makes stories boring.')
print('   The temp that makes stories vivid makes facts hallucinate.')


=== FACTUAL TASK: one correct answer ===
temp=0.0 should be reliable; temp=1.0 should be risky

  temp=0.0: 'Jupiter'
  temp=0.5: 'Jupiter'
  temp=1.0: 'Jupiter'

=== CREATIVE TASK: no single right answer ===
temp=0.0 = safe and generic. temp=1.0 = surprising and original

  temp=0.0: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window and watched the sunrise for the very first time — not because it was programmed to, but because it wanted to."
  temp=0.3: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window and watched the sunrise for the very first time — not because it was programmed to, but because it wanted to."
  temp=0.7: "The last robot on Earth woke up and, finding no one left to serve, quietly began planting flowers across the empty cities, because somewhere in its ancient programming it had learned that beauty needs no audience."
  temp=1.0: "The last robot on Earth woke up and, finding no one

,temperature,output
0,0.0,"""The last robot on Earth woke up and, finding ..."
1,0.3,"""The last robot on Earth woke up and, finding ..."
2,0.7,"""The last robot on Earth woke up and, finding ..."
3,1.0,"""The last robot on Earth woke up and, finding ..."



👉 The temp that makes facts reliable makes stories boring.
   The temp that makes stories vivid makes facts hallucinate.


In [31]:
# Prove determinism vs randomness on the same creative prompt
print('temp=0.0 — run 3 times (should be IDENTICAL):')
for i in range(3):
    text, _ = call(CREATIVE, temperature=0.0, max_tokens=60)
    print(f'  Run {i+1}: {text.strip()}')

print('\ntemp=1.0 — run 3 times (should VARY each time):')
for i in range(3):
    text, _ = call(CREATIVE, temperature=1.0, max_tokens=60)
    print(f'  Run {i+1}: {text.strip()}')

print('\n👉 temp=0.0 locks the output. temp=1.0 unlocks it.')


temp=0.0 — run 3 times (should be IDENTICAL):
  Run 1: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window and watched the sunrise for the very first time — not because it was programmed to, but because it wanted to."
  Run 2: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window and watched the sunrise for the very first time — not because it was programmed to, but because it wanted to."
  Run 3: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window and watched the sunrise for the very first time — not because it was programmed to, but because it wanted to."

temp=1.0 — run 3 times (should VARY each time):
  Run 1: "The last robot on Earth woke up and, finding no one left to serve, sat quietly by the window watching the sunrise, finally learning — in that profound, unhurried silence — what it meant to simply *be*."
  Run 2: "The last robot on Earth woke up and, fin

---
## Section 5 — `max_tokens` (Token Budget)

Hard upper limit on output length. When the model hits this limit it **stops mid-sentence**.

| Scenario | Budget | Result |
|---|---|---|
| Too small | 30 | Cut off mid-sentence |
| Adequate | 100 | Short summary |
| Comfortable | 300 | Full answer |

> **Rule of thumb:** 1 token ≈ 0.75 words. 100 tokens ≈ 1 paragraph. 500 tokens ≈ 1 page.
> **Cost impact:** you pay per token generated — don't set max_tokens=4096 for a one-liner task.

In [34]:
QUESTION = 'Explain how the internet works, step by step.'
results  = []
for budget in [30, 100, 300]:
    text, usage = call(QUESTION, max_tokens=budget, temperature=0.3)
    cut_off     = usage['output_tokens'] >= budget
    results.append({'max_tokens': budget, 'tokens_used': usage['output_tokens'],
                    'cut_off': '❌ YES' if cut_off else '✅ No'})
    print(f'\n--- max_tokens={budget} ---')
    print(text.strip())
    if cut_off:
        print('⚠️  HIT THE LIMIT — response was cut mid-generation')
print('\nSummary:')
display(pd.DataFrame(results))
print('👉 When tokens_used == max_tokens the model was interrupted, not finished.')


--- max_tokens=30 ---
# How the Internet Works: Step by Step

## 1. The Basic Concept
The internet is a **global network of computers** that
⚠️  HIT THE LIMIT — response was cut mid-generation

--- max_tokens=100 ---
# How the Internet Works: Step by Step

## 1. The Basic Concept
The internet is a **global network of interconnected computers** that communicate by following agreed-upon rules called **protocols**.

---

## 2. Your Device Connects to a Network

- Your phone or computer connects via **WiFi, ethernet, or cellular data**
- Your **Internet Service Provider (ISP)** gives your device an **IP address** — a unique numerical label
⚠️  HIT THE LIMIT — response was cut mid-generation

--- max_tokens=300 ---
# How the Internet Works: Step by Step

## 1. The Basic Concept
The internet is a **global network of interconnected computers** that communicate by following agreed-upon rules called **protocols**.

---

## 2. Your Device Connects to a Network

- Your phone or computer connects

,max_tokens,tokens_used,cut_off
0,30,30,❌ YES
1,100,100,❌ YES
2,300,300,❌ YES


👉 When tokens_used == max_tokens the model was interrupted, not finished.


---
## Section 6 — `top_p` (Nucleus Sampling)

At each step, rank all tokens by probability. `top_p` picks only from the tokens whose
cumulative probability adds up to `p`.

| Value | Tokens considered | Effect |
|---|---|---|
| `0.1` | Only top ~10% probability mass | Very conservative, common words only |
| `0.5` | Top 50% | Moderate vocabulary |
| `0.9` | Top 90% | Wide vocabulary, rarer words possible |

> ⚠️ Use `top_p` **OR** `temperature` — not both. When testing top_p, set `temperature=1.0` (neutral).

In [38]:
PROMPT  = 'Continue this sci-fi opening in exactly two sentences: "The last signal came from Mars on a Tuesday."'
results = []
for p in [0.1, 0.5, 0.9]:
    text, _ = call(PROMPT,  top_p=p, max_tokens=80)
    results.append({'top_p': p, 'output': text.strip()})
    print(f'\ntop_p={p}:')
    print(text.strip())
display(pd.DataFrame(results))
print('👉 top_p=0.1: safe, common phrasing. top_p=0.9: vivid, unexpected word choices.')


top_p=0.1:
"The last signal came from Mars on a Tuesday."

It was a child's voice, singing a nursery rhyme that no one on the red planet should have known. By Friday, every telescope on Earth was pointed at the sky, and every government on Earth was lying about what they saw.

top_p=0.5:
"The last signal came from Mars on a Tuesday."

It was a child's voice, singing a nursery rhyme that no one on the red planet should have known. By Friday, every telescope on Earth was pointed at the coordinates where Mars used to be.

top_p=0.9:
"The last signal came from Mars on a Tuesday."

It was a child's voice, singing a nursery rhyme that no one had taught the colonists. By Friday, every telescope on Earth was pointed at the red planet, watching the lights go out one by one.


,top_p,output
0,0.1,"""The last signal came from Mars on a Tuesday.""..."
1,0.5,"""The last signal came from Mars on a Tuesday.""..."
2,0.9,"""The last signal came from Mars on a Tuesday.""..."


👉 top_p=0.1: safe, common phrasing. top_p=0.9: vivid, unexpected word choices.


---
## Section 7 — `top_k`

At each step, only consider the **k most probable tokens** — discard all others.

| Value | Effect | Use case |
|---|---|---|
| `1` | Greedy — always picks #1. Fully deterministic. | Structured output, code |
| `10` | Top 10 candidates only | Controlled, low-variation |
| `250` | 250 candidates | Creative, diverse |

> `top_k=1` is more aggressive than `temperature=0.0` — it completely eliminates randomness.

In [40]:
PROMPT  = 'Write the opening line of a poem about rain. Just one line.'
results = []
for k in [1, 10, 250]:
    runs = []
    for _ in range(3):
        text, _ = call(PROMPT, temperature=1.0, top_k=k, max_tokens=30)
        runs.append(text.strip())
    unique = len(set(runs))
    results.append({'top_k': k, 'run_1': runs[0], 'run_2': runs[1],
                    'run_3': runs[2], 'unique_outputs': f'{unique}/3'})
    print(f'\ntop_k={k} — unique outputs across 3 runs: {unique}/3')
    for i, r in enumerate(runs, 1):
        print(f'  Run {i}: {r}')
display(pd.DataFrame(results))
print('👉 top_k=1: identical every time. top_k=250: different every time.')


top_k=1 — unique outputs across 3 runs: 2/3
  Run 1: The rain arrives like a confession, soft and overdue.
  Run 2: The rain arrives like a confession, soft and overdue.
  Run 3: The rain arrives like a confession, soft and long overdue.

top_k=10 — unique outputs across 3 runs: 3/3
  Run 1: The rain arrives like a confession, soft and overdue.
  Run 2: The rain arrives like a letter no one asked for, ink bleeding through the page.
  Run 3: The rain arrives like a letter no one asked for, wet with everything unsaid.

top_k=250 — unique outputs across 3 runs: 3/3
  Run 1: The rain arrives like a confession, slow and inevitable.
  Run 2: Here is one opening line:

"The rain arrives like a secret the clouds could no longer keep."
  Run 3: The rain arrives like a secret the clouds could no longer keep.


,top_k,run_1,run_2,run_3,unique_outputs
0,1,"The rain arrives like a confession, soft and o...","The rain arrives like a confession, soft and o...","The rain arrives like a confession, soft and l...",2/3
1,10,"The rain arrives like a confession, soft and o...",The rain arrives like a letter no one asked fo...,The rain arrives like a letter no one asked fo...,3/3
2,250,"The rain arrives like a confession, slow and i...","Here is one opening line:\n\n""The rain arrives...",The rain arrives like a secret the clouds coul...,3/3


👉 top_k=1: identical every time. top_k=250: different every time.


---
## Section 8 — `stop_sequences`

Tell the model to **halt immediately** when it outputs a specific string.
The stop token itself is NOT included in the output.

| Use case | stop_sequences value |
|---|---|
| Get only first N list items | `['\n4.']` stops after item 3 |
| Stop at section header | `['###']` |
| Extract one JSON field | `[',']` or `['}']` |

> ❌ Without stop: model generates all 10 items (wasted tokens, wasted money)
> ✅ With stop: precise control over where output ends

In [41]:
LIST_PROMPT = 'List 10 programming languages, numbered 1-10, one per line, with a one-sentence description each.'
print('❌ Without stop_sequences — all 10 items:\n')
text, usage = call(LIST_PROMPT, max_tokens=600, temperature=0.3)
print(text)
print(f'\nTokens used: {usage["output_tokens"]}')

❌ Without stop_sequences — all 10 items:

Here are 10 programming languages:

1. **Python** - A versatile, beginner-friendly language known for its readable syntax, widely used in data science, AI, and web development.
2. **JavaScript** - The primary language of the web, enabling interactive and dynamic content in browsers and server-side applications via Node.js.
3. **Java** - A robust, object-oriented language designed for portability, running on the Java Virtual Machine across many platforms.
4. **C** - A foundational low-level language that provides direct memory control and serves as the basis for many modern languages.
5. **C++** - An extension of C that adds object-oriented features, commonly used in game development, systems programming, and performance-critical applications.
6. **Rust** - A modern systems programming language focused on memory safety and performance without the need for a garbage collector.
7. **Go** - A statically typed language developed by Google, designed 

In [43]:
# ✅ WITH stop_sequence — stops after item 3
print('✅ With stop_sequences=["\\n4."] — stops after item 3:\n')
text, usage = call(LIST_PROMPT, max_tokens=600, temperature=0.3, stop_sequences=['4.'])
print(text)
print(f'\nTokens used: {usage["output_tokens"]}')
print('👉 Drastically fewer tokens — you paid only for what you needed.')

✅ With stop_sequences=["\n4."] — stops after item 3:

Here are 10 programming languages:

1. **Python** - A versatile, beginner-friendly language known for its readable syntax, widely used in data science, AI, and web development.
2. **JavaScript** - The primary language of the web, enabling interactive and dynamic content in browsers and server-side applications via Node.js.
3. **Java** - A robust, object-oriented language designed to be platform-independent, commonly used in enterprise applications and Android development.


Tokens used: 106
👉 Drastically fewer tokens — you paid only for what you needed.


---
## Section 9 — Multi-turn Messages (Conversation History)

The `messages` array is how context works. Each call is stateless —
you must pass the full conversation history yourself.

> ❌ Without history: model forgets everything said before
> ✅ With history: model remembers and builds on prior turns

In [45]:
# ❌ WITHOUT conversation history — model has no memory
print('❌ Without history — two separate calls:\n')

text1, _ = call('My name is Alice and I love astronomy.')
print(f'Turn 1 → {text1.strip()[:80]}')

text2, _ = call('What is my name and what do I love?')  # fresh call, no history
print(f'Turn 2 → {text2.strip()}')
print('❌ Model has no idea who Alice is — each call is a blank slate.')

❌ Without history — two separate calls:

Turn 1 → Hi Alice! It's great to meet you! Astronomy is a fascinating field. There's so m
Turn 2 → I don't have any information about you personally. I don't know your name or what you love. I only know what you share with me in our conversation.

Would you like to tell me about yourself? 😊
❌ Model has no idea who Alice is — each call is a blank slate.


In [46]:
# ✅ WITH conversation history — messages array carries context
print('✅ With full history in messages array:\n')

history = [
    {'role': 'user',      'content': 'My name is Alice and I love astronomy.'},
    {'role': 'assistant', 'content': 'Nice to meet you, Alice! Astronomy is a fascinating field.'},
    {'role': 'user',      'content': 'What is my name and what do I love?'},
]

text, _ = call('', messages=history, max_tokens=80)
print(f'Model response: {text.strip()}')
print('✅ Model remembered because WE passed the history. This is how all chatbots work.')

✅ With full history in messages array:

Model response: Your name is Alice and you love astronomy! 😊
✅ Model remembered because WE passed the history. This is how all chatbots work.


In [47]:
# Build a multi-turn conversation programmatically
print('Multi-turn conversation builder:\n')
conv_history = []

def chat(user_message, system=None):
    conv_history.append({'role': 'user', 'content': user_message})
    text, _ = call('', messages=conv_history, system=system, max_tokens=150)
    conv_history.append({'role': 'assistant', 'content': text})
    return text.strip()

SYSTEM = 'You are a helpful Python tutor. Keep answers to 2 sentences.'

turns = [
    'What is a list in Python?',
    'How is it different from a tuple?',
    'Can you give me a one-line example of each?',
]
for user_msg in turns:
    reply = chat(user_msg, system=SYSTEM)
    print(f'User:    {user_msg}')
    print(f'Claude:  {reply}')
    print()

Multi-turn conversation builder:

User:    What is a list in Python?
Claude:  A **list** in Python is an ordered, mutable collection of items that can hold elements of different data types, defined using square brackets (e.g., `my_list = [1, "hello", 3.14]`). You can add, remove, and modify elements in a list after it's created.

User:    How is it different from a tuple?
Claude:  The main difference is that a **tuple** is **immutable**, meaning its elements cannot be changed after creation, and it's defined using parentheses (e.g., `my_tuple = (1, "hello", 3.14)`). Lists are better for collections that need to change, while tuples are better for fixed data and are slightly faster and more memory-efficient.

User:    Can you give me a one-line example of each?
Claude:  Sure! Here's a quick example of each:

- **List:** `my_list = [1, 2, 3]` → you can do `my_list[0] = 10` to change an element.
- **Tuple:** `my_tuple = (1, 2, 3)` → trying `my_tuple[0] = 10` would raise a `TypeError`.



---
## Section 10 — Tool Use (Function Calling)

You define a tool with a schema. The model decides **when to call it** and
returns structured arguments — you execute the function and feed the result back.

```
User asks → Model decides to call tool → Returns {name, args}
  → You run the real function → Feed result back → Model gives final answer
```

> ❌ Without tools: model makes up data (hallucination)
> ✅ With tools: model requests real data, answer is grounded in facts

In [ ]:
# ❌ WITHOUT tool — model makes up the stock price
print('❌ Without tool — model fabricates the answer:\n')
text, _ = call("What is NVIDIA's current stock price?", temperature=0.0)
print(text)
print('❌ This is a hallucination — the model cannot know real-time data.')

In [51]:
# ✅ WITH tool — model requests real data via a defined schema
import yfinance as yf

# Define the tool schema
tools = [{
    'name': 'get_stock_price',
    'description': 'Get the current stock price for a given ticker symbol.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'ticker': {'type': 'string', 'description': 'Stock ticker, e.g. NVDA'},
        },
        'required': ['ticker']
    }
}]

# Step 1: Ask the model
body = json.dumps({
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 200,
    'tools': tools,
    'messages': [{'role': 'user', 'content': "What is NVIDIA's current stock price?"}]
})
resp   = client.invoke_model(modelId=MODEL, body=body)
result = json.loads(resp['body'].read())

print('Step 1 — Model response type:', result['stop_reason'])
tool_call = next(b for b in result['content'] if b['type'] == 'tool_use')
print('Model wants to call:', tool_call['name'])
print('With arguments:', tool_call['input'])

# Step 2: Simulate running the real function
def get_stock_price(ticker):
    stock = yf.Ticker(ticker)
    price = stock.fast_info['last_price']
    return {
        'ticker':   ticker,
        'price':    round(price, 2),
        'currency': stock.fast_info['currency'],
        'as_of':    'live',
    }
    #return {'ticker': ticker, 'price': 875.42, 'currency': 'USD', 'as_of': '2025-04-25'}

tool_result = get_stock_price(**tool_call['input'])
print('\nStep 2 — Tool returned:', tool_result)

# Step 3: Feed result back to model for final answer
body2 = json.dumps({
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 150,
    'tools': tools,
    'messages': [
        {'role': 'user', 'content': "What is NVIDIA's current stock price?"},
        {'role': 'assistant', 'content': result['content']},
        {'role': 'user', 'content': [{
            'type': 'tool_result',
            'tool_use_id': tool_call['id'],
            'content': json.dumps(tool_result)
        }]}
    ]
})
resp2   = client.invoke_model(modelId=MODEL, body=body2)
result2 = json.loads(resp2['body'].read())
print('\nStep 3 — Final answer:')
print(result2['content'][0]['text'])
print('✅ Model used a tool instead of guessing — answer is grounded in real data.')

Step 1 — Model response type: tool_use
Model wants to call: get_stock_price
With arguments: {'ticker': 'NVDA'}

Step 2 — Tool returned: {'ticker': 'NVDA', 'price': 208.27, 'currency': 'USD', 'as_of': 'live'}

Step 3 — Final answer:
NVIDIA's (NVDA) current stock price is **$208.27 USD**. Would you like to know anything else about NVIDIA or any other stocks?
✅ Model used a tool instead of guessing — answer is grounded in real data.


---
## Section 11 — Penalty Parameters (Concept)

Claude does not expose penalty parameters directly, but they are important in other models.

| Parameter | Supported in | Effect |
|---|---|---|
| `frequency_penalty` | OpenAI, some Mistral | Penalises tokens in proportion to how often they have appeared — reduces repetition |
| `presence_penalty` | OpenAI | Penalises any token that has appeared at all — encourages topic diversity |
| `repetition_penalty` | Llama, Mistral (Bedrock) | Multiplier: >1.0 discourages repeating tokens |

> Claude controls repetition through `temperature` + `top_p` + good prompting.
> We simulate the penalty effect below by comparing outputs with and without explicit anti-repetition instructions.

In [54]:
# Demonstrate repetition problem and how to mitigate it in Claude
REPETITION_PROMPT = 'Write 5 marketing bullet points for a fitness app.'

# ❌ High temperature without guidance — can repeat synonyms/phrases
print('❌ High temp, no anti-repetition guidance:\n')
text, _ = call(REPETITION_PROMPT, temperature=1.0, max_tokens=200)
print(text)

# ✅ Explicit instruction — equivalent to presence_penalty in other models
print('\n✅ Same prompt with explicit uniqueness constraint:\n')
text, _ = call(
    REPETITION_PROMPT,
    system='Each bullet point must use completely different words and cover a distinct benefit. No repeated words across bullets.',
    temperature=1.0, max_tokens=200
)
print(text)

print('\n👉 Claude has no repetition_penalty param.')
print('   A system prompt constraint achieves the same effect.')


❌ High temp, no anti-repetition guidance:

Here are 5 marketing bullet points for a fitness app:

• **Personalized Workouts at Your Fingertips** — Get custom exercise plans tailored to your fitness level, goals, and schedule, so every workout counts.

• **Track Your Progress in Real Time** — Monitor calories burned, steps taken, and personal records with intuitive dashboards that keep you motivated and on track.

• **500+ Guided Workouts for Every Level** — From beginner yoga to advanced HIIT training, access a full library of video-guided sessions you can do anywhere, anytime.

• **Stay Accountable with a Built-In Community** — Connect with friends, join challenges, and celebrate milestones together in a supportive fitness community.

• **No Gym Required** — Whether you're at home, traveling, or outdoors, get a world-class workout with nothing more than your phone and your determination.

✅ Same prompt with explicit uniqueness constraint:

Here are 5 marketing bullet points for a fitn

---
## Section 12 — Logging

> Every call we made above generated tokens, cost money, and took time.
> Without logging, all that data is gone. With it, you can optimise.

In [55]:
test_prompts = ['What is machine learning?','Explain neural networks briefly.',
                'What is gradient descent?','Define overfitting.','What is a transformer?']

print('❌ 5 calls, no logging:\n')
for p in test_prompts:
    text, _ = call(p, max_tokens=80)
    print(f'  Q: {p}\n  A: {text.strip()[:80]}\n')
print('❌ Which call used most tokens? Which was slowest? What did it cost? → Cannot answer.')

❌ 5 calls, no logging:

  Q: What is machine learning?
  A: # Machine Learning

Machine learning (ML) is a branch of **artificial intelligen

  Q: Explain neural networks briefly.
  A: # Neural Networks

## What They Are
Neural networks are computational models **i

  Q: What is gradient descent?
  A: # Gradient Descent

Gradient descent is an **optimization algorithm** used to mi

  Q: Define overfitting.
  A: ## Overfitting

**Overfitting** occurs when a machine learning model learns the 

  Q: What is a transformer?
  A: # Transformer

The term "transformer" can refer to several things depending on c

❌ Which call used most tokens? Which was slowest? What did it cost? → Cannot answer.


In [56]:
PRICE    = {'input': 3.00, 'output': 15.00}  # Claude Sonnet 4.6 approx per 1M tokens
call_log = []

def logged_call(prompt, **kwargs):
    start       = time.time()
    text, usage = call(prompt, **kwargs)
    t_in        = usage['input_tokens']
    t_out       = usage['output_tokens']
    cost        = (t_in * PRICE['input'] + t_out * PRICE['output']) / 1_000_000
    call_log.append({'prompt': prompt[:40]+'...', 'in_tok': t_in, 'out_tok': t_out,
                     'latency_ms': round((time.time()-start)*1000), 'cost_usd': round(cost,6)})
    return text

for p in test_prompts:
    logged_call(p, max_tokens=80)

log_df = pd.DataFrame(call_log)
display(log_df)
print(f'Most tokens:  {log_df.loc[log_df["out_tok"].idxmax(), "prompt"]}')
print(f'Slowest call: {log_df.loc[log_df["latency_ms"].idxmax(), "prompt"]}')
print(f'Total cost:   ${log_df["cost_usd"].sum():.6f}')
print(f'Avg latency:  {log_df["latency_ms"].mean():.0f}ms')

,prompt,in_tok,out_tok,latency_ms,cost_usd
0,What is machine learning?...,12,80,11342,0.001236
1,Explain neural networks briefly....,13,80,2717,0.001239
2,What is gradient descent?...,12,80,2201,0.001236
3,Define overfitting....,12,79,2924,0.001221
4,What is a transformer?...,12,80,2896,0.001236


Most tokens:  What is machine learning?...
Slowest call: What is machine learning?...
Total cost:   $0.006168
Avg latency:  4416ms


---
## ✅ Complete Parameter Reference

| Parameter | Controls | Tip |
|---|---|---|
| `system` | Persona & behaviour | Every production app needs one |
| Streaming | UX responsiveness | Always on in chat apps |
| `temperature` | Output randomness | 0.0 for facts, 0.7 for chat, 1.0 for creativity |
| `max_tokens` | Output length budget | 1 token ≈ 0.75 words — don't over-provision |
| `top_p` | Vocabulary width | Use instead of temperature, not alongside |
| `top_k` | Candidate pool size | 1 = fully deterministic |
| `stop_sequences` | Where to halt | Precision output control |
| `messages` | Conversation history | You manage state — pass full history each call |
| `tools` | Function calling | Grounds model in real data, prevents hallucination |
| `repetition_penalty` | Repetition (Llama/Mistral) | Use system prompt as equivalent in Claude |

**Next →** `02_rapid_prototyping.ipynb`